# AI & automation context (NOT an economic shock)

Uses only explicitly supplied real annual observations; distinguishes country and sector coverage.


In [ ]:
from pathlib import Path
import sys
for loc in [Path.cwd(),Path.cwd()/'notebooks',Path.cwd()/'research_expansion'/'notebooks']:
    if (loc/'common.py').exists():
        sys.path.insert(0,str(loc.resolve()));break
from common import root, output


In [ ]:
import pandas as pd, numpy as np
path=root()/'data'/'processed'/'ai_indicators.csv'
if not path.exists():
    output('ai_transition',{'model':'AI adoption structural context','status':'skipped','reason':'Missing observed annual AI series: data/processed/ai_indicators.csv','limitations':['AI transition is not a dated causal shock','Never assume country-level investment represents specific manufacturing industries']})
else:
    ai=pd.read_csv(path,dtype={'country':'string','industry_code':'string','indicator_id':'string'})
    required={'year','country','industry_code','indicator_id','value','unit','source_url','definition','geography_level'}
    if not required.issubset(ai.columns):
        raise ValueError('AI file missing: '+str(sorted(required-set(ai.columns))))
    ai['year']=pd.to_numeric(ai.year,errors='coerce');ai['value']=pd.to_numeric(ai.value,errors='coerce')
    ai=ai.dropna(subset=['year','country','indicator_id','value','unit','source_url','definition','geography_level']).copy()
    ai=ai[ai.year.between(2021,2026)]
    if ai.empty:
        output('ai_transition',{'status':'skipped','reason':'No valid observed AI indicator rows during 2021–2026'})
    else:
        ai['industry_code']=ai['industry_code'].fillna('COUNTRY_LEVEL').replace('', 'COUNTRY_LEVEL')
        ai=ai.sort_values('year')
        gcols=['country','industry_code','indicator_id','unit','definition','geography_level']
        summary=ai.groupby(gcols,dropna=False).agg(n_years=('year','nunique'),first=('year','min'),last=('year','max'),first_value=('value','first'),last_value=('value','last')).reset_index()
        summary['value_change']=np.where(summary.n_years>=2,summary.last_value-summary.first_value,np.nan)
        output('ai_transition',{'model':'AI adoption structural context','status':'descriptive',
           'n_observations':len(ai),'series':summary.astype(object).where(pd.notna(summary),None).to_dict('records'),
           'limitations':['Different AI measurement methods and geographic granularity MUST NOT be merged','Changes are in the raw indicator units; percentage-point only when indicator is adoption percent','No inference of AI-caused productivity or firm survival']})
        display(summary)

if path.exists() and not ai.empty:
    import matplotlib.pyplot as plt
    for _, obs in ai.groupby(gcols,dropna=False):
        plt.plot(obs.year,obs.value,'o-',label=str(obs.country.iloc[0]))
    plt.xticks(sorted(ai.year.unique()))
    plt.ylabel('Percent of surveyed enterprises')
    plt.title('Recorded AI adoption observations')
    plt.show()
